In [4]:
!pip install pyspark==3.5.0

spark = SparkSession.builder \
    .appName("Top_Programming_Languages") \
    .getOrCreate()

print("Spark запущен")

Spark запущен


In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
import pyspark.sql.functions as F

df_langs = spark.read.csv("programming-languages.csv", header=True)
first_col_name = df_langs.columns[0]
df_langs_clean = df_langs.select(F.trim(F.lower(F.col(first_col_name))).alias("language"))

df_posts = spark.read.text("posts_sample.xml")

df_rows = df_posts.filter(F.col("value").contains("<row") & F.col("value").contains('PostTypeId="1"'))

df_parsed = df_rows.withColumn("year", F.regexp_extract("value", r'CreationDate="(\d{4})', 1).cast("int")) \
                   .withColumn("tags_raw", F.regexp_extract("value", r'Tags="([^"]+)"', 1))

# 2010 - 2020
df_period = df_parsed.filter((F.col("year") >= 2010) & (F.col("year") <= 2020))

# раскодировка тегов
df_decoded = df_period.withColumn("tags_decoded", F.regexp_replace("tags_raw", "&lt;", "<")) \
                      .withColumn("tags_decoded", F.regexp_replace("tags_decoded", "&gt;", ">"))

df_tags_array = df_decoded.withColumn("tags_clean", F.regexp_replace("tags_decoded", r'^<|>$', "")) \
                          .withColumn("tag_list", F.split("tags_clean", "><"))

df_exploded = df_tags_array.select("year", F.explode("tag_list").alias("tag_name"))
# чистим от случайных пробелов и переводим в ниж регистр
df_exploded = df_exploded.withColumn("tag_name", F.trim(F.lower(F.col("tag_name"))))

df_matched = df_exploded.join(df_langs_clean, df_exploded.tag_name == df_langs_clean.language, "inner")

# группируем, считаем количество упоминаний
df_counts = df_matched.groupBy("year", "language").count()

# топ-10 для каждого года с помощью оконной функции
windowSpec = Window.partitionBy("year").orderBy(F.desc("count"))
df_ranked = df_counts.withColumn("rank", F.row_number().over(windowSpec))

# топ-10
df_top10 = df_ranked.filter(F.col("rank") <= 10).orderBy("year", "rank")

print("ДИНАМИКА ПОПУЛЯРНОСТИ ЯЗЫКОВ ПРОГРАММИРОВАНИЯ (2010-2020)")
df_top10.show(110, truncate=False) # 110 строк, чтобы поместились все 11 лет по 10 языков

parquet_path = "top10_languages_2010_2020.parquet"
df_top10.write.mode("overwrite").parquet(parquet_path)
print(f"Отчет сохранен в формате Parquet: {parquet_path}")

ДИНАМИКА ПОПУЛЯРНОСТИ ЯЗЫКОВ ПРОГРАММИРОВАНИЯ (2010-2020)
+----+-----------+-----+----+
|year|language   |count|rank|
+----+-----------+-----+----+
|2010|java       |52   |1   |
|2010|php        |46   |2   |
|2010|javascript |44   |3   |
|2010|python     |26   |4   |
|2010|objective-c|23   |5   |
|2010|c          |20   |6   |
|2010|ruby       |12   |7   |
|2010|delphi     |8    |8   |
|2010|applescript|3    |9   |
|2010|r          |3    |10  |
|2011|php        |102  |1   |
|2011|java       |93   |2   |
|2011|javascript |83   |3   |
|2011|python     |37   |4   |
|2011|objective-c|34   |5   |
|2011|c          |24   |6   |
|2011|ruby       |20   |7   |
|2011|perl       |9    |8   |
|2011|delphi     |8    |9   |
|2011|bash       |7    |10  |
|2012|php        |154  |1   |
|2012|javascript |132  |2   |
|2012|java       |124  |3   |
|2012|python     |69   |4   |
|2012|objective-c|45   |5   |
|2012|ruby       |27   |6   |
|2012|c          |27   |7   |
|2012|bash       |10   |8   |
|2012|r     

In [6]:
# проверка, чтение данных из Apache Parquet
print("ЧТЕНИЕ ОТЧЕТА ИЗ СОХРАНЕННОГО PARQUET ФАЙЛА")

# считываем папку Parquet обратно в DataFrame
df_from_parquet = spark.read.parquet("top10_languages_2010_2020.parquet")

# перед выводом снова отсортируем данные по году и рангу
df_from_parquet_sorted = df_from_parquet.orderBy("year", "rank")

# первые 30 строк для наглядности
df_from_parquet_sorted.show(30, truncate=False)

print("Данные прочитаны из Parquet.")

ЧТЕНИЕ ОТЧЕТА ИЗ СОХРАНЕННОГО PARQUET ФАЙЛА
+----+-----------+-----+----+
|year|language   |count|rank|
+----+-----------+-----+----+
|2010|java       |52   |1   |
|2010|php        |46   |2   |
|2010|javascript |44   |3   |
|2010|python     |26   |4   |
|2010|objective-c|23   |5   |
|2010|c          |20   |6   |
|2010|ruby       |12   |7   |
|2010|delphi     |8    |8   |
|2010|applescript|3    |9   |
|2010|r          |3    |10  |
|2011|php        |102  |1   |
|2011|java       |93   |2   |
|2011|javascript |83   |3   |
|2011|python     |37   |4   |
|2011|objective-c|34   |5   |
|2011|c          |24   |6   |
|2011|ruby       |20   |7   |
|2011|perl       |9    |8   |
|2011|delphi     |8    |9   |
|2011|bash       |7    |10  |
|2012|php        |154  |1   |
|2012|javascript |132  |2   |
|2012|java       |124  |3   |
|2012|python     |69   |4   |
|2012|objective-c|45   |5   |
|2012|ruby       |27   |6   |
|2012|c          |27   |7   |
|2012|bash       |10   |8   |
|2012|r          |9    |9 